# Similarities between FORUM ranking policies

This notebook treats each of the **90 FORUM policy bundles** as one point: 15 orderings (including random) × 3 reply modes × 2 pin states. A point is represented by its story-averaged FORUM values on the 41 scalar features used by the regression/ML ranking models.

The notebook constructs two equally important spaces—**top 10** and **full list**—and clusters each independently. It reads the already-calculated FORUM values from the FORUM policy-score product; it does not recalculate rankings or FORUM trajectories.

Raw BGE dimensions and the newer FORUM policy outcomes are excluded. The analysis deliberately starts with unstandardized FORUM axes.

In [ ]:
from pathlib import Path
import json

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.cluster.hierarchy import cut_tree, dendrogram, leaves_list, linkage
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score

from commentgap_analysis.forum_scores import FORUM_OUTCOMES, REGRESSION_FEATURE_COLUMNS

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 120
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'commentgap_analysis').exists() and (REPO_ROOT.parent / 'commentgap_analysis').exists():
    REPO_ROOT = REPO_ROOT.parent
SCORES_PATH = REPO_ROOT / 'model_output/selection_2025/forum_ranking_analysis/policy_scores/policy_scores.parquet'
OUTPUT_ROOT = REPO_ROOT / 'model_output/selection_2025/ranking_policy_similarity'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Main analysis configuration.
FEATURE_SPACE = 'regression'  # 'regression' or 'forum'
STANDARDIZE = False           # z-score each selected feature across policy bundles
if FEATURE_SPACE not in {'regression', 'forum'}:
    raise ValueError("FEATURE_SPACE must be 'regression' or 'forum'.")
FEATURES = list(REGRESSION_FEATURE_COLUMNS if FEATURE_SPACE == 'regression' else FORUM_OUTCOMES)
SPACE_LABEL = 'regression features' if FEATURE_SPACE == 'regression' else 'FORUM outcomes'
SCALE_LABEL = 'standardized' if STANDARDIZE else 'unstandardized'

DEPTHS = ('top10', 'full')
MAX_K = 12
USER_CLUSTER_K = {'top10': None, 'full': None}  # Optional manual overrides.
print(f'{len(FEATURES)} {SPACE_LABEL}; {SCALE_LABEL}')

## Sample and existing FORUM product

In the FORUM code, the **primary sample** means every eligible discussion with at least 11 comments. The separate large-thread sensitivity subset requires at least 100 comments. Here both top-10 and full-list spaces use all eligible scored discussions; neither depth is treated as a sensitivity analysis.

DuckDB filters and aggregates the existing Parquet product before moving data into pandas. The expected result is only 90 × 12 × 2 rows.

In [ ]:
if not SCORES_PATH.exists():
    raise FileNotFoundError(SCORES_PATH)
cache_candidates = (
    [OUTPUT_ROOT / 'policy_feature_mean_regression.csv',
     OUTPUT_ROOT / 'policy_feature_mean_forum.csv']
    if FEATURE_SPACE == 'regression'
    else [OUTPUT_ROOT / 'policy_feature_mean_forum_outcomes.csv']
)
aggregate_cache = next((path for path in cache_candidates if path.exists()), cache_candidates[0])
expected_rows = 90 * len(FEATURES) * len(DEPTHS)
cache_is_current = False
if aggregate_cache.exists():
    cached = pd.read_csv(aggregate_cache)
    cache_is_current = (
        len(cached) == expected_rows
        and cached.policy_id.nunique() == 90
        and set(cached.outcome) == set(FEATURES)
        and set(cached.depth) == set(DEPTHS)
    )
    if cache_is_current:
        feature_means = cached
        print(f'Loaded current {SPACE_LABEL} cache: {aggregate_cache.relative_to(REPO_ROOT)}')
if not cache_is_current:
    feature_sql = ', '.join("'" + feature.replace("'", "''") + "'" for feature in FEATURES)
    scores_sql_path = str(SCORES_PATH).replace("'", "''")
    query = f'''
    SELECT policy_id, ordering, reply_mode, pinned, deployable, outcome, depth,
           AVG(forum) AS mean_forum, COUNT(DISTINCT story_id) AS n_stories
    FROM read_parquet('{scores_sql_path}')
    WHERE outcome IN ({feature_sql}) AND depth IN ('top10', 'full')
    GROUP BY policy_id, ordering, reply_mode, pinned, deployable, outcome, depth
    ORDER BY depth, policy_id, outcome
    '''
    feature_means = duckdb.sql(query).df()

if len(feature_means) != expected_rows:
    raise ValueError(f'Expected {expected_rows:,} aggregated rows, found {len(feature_means):,}.')
if feature_means.policy_id.nunique() != 90:
    raise ValueError('The policy product does not contain all 90 policy bundles.')
if set(feature_means.outcome) != set(FEATURES):
    raise ValueError(f'The available axes do not match the selected {SPACE_LABEL}.')
coverage = feature_means.groupby(['depth', 'policy_id', 'outcome']).n_stories.first()
if coverage.nunique() != 1:
    raise ValueError('Policy-feature cells do not have identical story coverage.')

policy_metadata = feature_means[
    ['policy_id', 'ordering', 'reply_mode', 'pinned', 'deployable']
].drop_duplicates().sort_values('policy_id').reset_index(drop=True)
policy_metadata['pin_state'] = np.where(policy_metadata.pinned, 'pinned', 'unpinned')
policy_metadata['interface'] = policy_metadata.reply_mode + ' · ' + policy_metadata.pin_state

display(policy_metadata.groupby(['ordering', 'reply_mode', 'pinned'], dropna=False).size().to_frame('bundles'))
print(f"{feature_means.policy_id.nunique()} bundles; {feature_means.outcome.nunique()} axes; "
      f"{int(coverage.iloc[0]):,} stories per policy-feature-depth cell.")

## Construct the two raw FORUM spaces

Rows are policy bundles and columns are model features. Values are article-weighted mean FORUM scores. No z-scoring or variance rescaling is applied. Because every FORUM axis has the same theoretical range, Euclidean distance has a direct interpretation here—but features with more observed cross-policy variation will contribute more to distance.

In [ ]:
matrices = {}
for depth in DEPTHS:
    matrix = feature_means[feature_means.depth.eq(depth)].pivot(
        index='policy_id', columns='outcome', values='mean_forum'
    )
    matrix = matrix.reindex(index=policy_metadata.policy_id, columns=FEATURES)
    if matrix.isna().any().any() or matrix.shape != (90, len(FEATURES)):
        raise ValueError(f'Incomplete {depth} FORUM matrix: {matrix.shape}')
    matrices[depth] = matrix

if STANDARDIZE:
    matrices = {depth: (matrix - matrix.mean()) / matrix.std(ddof=1) for depth, matrix in matrices.items()}

feature_dispersion = pd.DataFrame({
    depth: matrices[depth].std(axis=0) for depth in DEPTHS
})
feature_dispersion['mean_sd'] = feature_dispersion.mean(axis=1)
feature_order = feature_dispersion.sort_values('mean_sd', ascending=False).index.tolist()
display(feature_dispersion.sort_values('mean_sd', ascending=False).head(15))
print('Matrix shapes:', {depth: matrix.shape for depth, matrix in matrices.items()})

## Hierarchical clustering and the number of clusters

Ward hierarchical clustering is applied separately to the two raw Euclidean spaces. Ward linkage asks which merge causes the smallest increase in within-cluster sum of squares, which matches the geometry of these feature profiles.

The dendrogram remains the primary result. The automatic cut maximizes silhouette score over `k=2..12`; it is a descriptive aid, not evidence that the policies possess one true number of clusters. Set `USER_CLUSTER_K` above to impose a more interpretable cut after inspecting the hierarchy.

In [ ]:
spaces = {}
selection_rows = []
for depth in DEPTHS:
    matrix = matrices[depth]
    condensed = pdist(matrix.to_numpy(), metric='euclidean')
    distance = squareform(condensed)
    tree = linkage(condensed, method='ward')
    depth_rows = []
    for k in range(2, min(MAX_K, len(matrix) - 1) + 1):
        labels = cut_tree(tree, n_clusters=k).reshape(-1) + 1
        depth_rows.append({
            'depth': depth, 'k': k,
            'silhouette': silhouette_score(distance, labels, metric='precomputed'),
            'smallest_cluster': int(pd.Series(labels).value_counts().min()),
            'largest_cluster': int(pd.Series(labels).value_counts().max()),
        })
    depth_selection = pd.DataFrame(depth_rows)
    chosen_k = USER_CLUSTER_K[depth]
    if chosen_k is None:
        chosen_k = int(depth_selection.loc[depth_selection.silhouette.idxmax(), 'k'])
    labels = cut_tree(tree, n_clusters=chosen_k).reshape(-1) + 1
    spaces[depth] = {
        'matrix': matrix, 'distance': distance, 'linkage': tree,
        'labels': labels, 'k': chosen_k,
    }
    selection_rows.extend(depth_rows)

cluster_selection = pd.DataFrame(selection_rows)
display(cluster_selection.pivot(index='k', columns='depth', values='silhouette').style.format('{:.3f}'))
print('Selected cuts:', {depth: spaces[depth]['k'] for depth in DEPTHS})

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 14), gridspec_kw={'width_ratios': [1.25, 1]})
for row, depth in enumerate(DEPTHS):
    selection = cluster_selection[cluster_selection.depth.eq(depth)]
    axes[row, 0].plot(selection.k, selection.silhouette, marker='o')
    axes[row, 0].axvline(spaces[depth]['k'], color='black', linestyle='--', linewidth=1)
    axes[row, 0].set(
        title=f'{depth}: silhouette across dendrogram cuts', xlabel='Number of clusters',
        ylabel='Silhouette score', xticks=selection.k,
    )
    dendrogram(spaces[depth]['linkage'], no_labels=True, ax=axes[row, 1])
    axes[row, 1].set(
        title=f'{depth}: Ward hierarchy of all 90 bundles',
        xlabel='Policy bundles', ylabel='Ward distance',
    )
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / 'raw_cluster_diagnostics.png', dpi=180, bbox_inches='tight')
plt.show()

## Raw FORUM feature maps

These are the requested policy × model-feature spaces. Rows follow each depth's own policy dendrogram. Columns use one common order—descending average cross-policy dispersion—so top-10 and full-list patterns can be compared directly. Colour represents raw mean FORUM, not a standardized value.

In [ ]:
global_limit = max(np.abs(matrix.to_numpy()).max() for matrix in matrices.values())
for depth in DEPTHS:
    row_order = leaves_list(spaces[depth]['linkage'])
    ordered = matrices[depth].iloc[row_order][feature_order]
    fig, ax = plt.subplots(figsize=(20, 25))
    sns.heatmap(
        ordered, cmap='vlag', center=0, vmin=-global_limit, vmax=global_limit,
        xticklabels=[name.replace('_', ' ') for name in ordered.columns],
        yticklabels=True, cbar_kws={'label': 'Raw article-weighted mean FORUM'}, ax=ax,
    )
    ax.set(
        title=f'{depth}: all 90 policy bundles × 41 regression/ML features',
        xlabel='Regression / ML feature', ylabel='Policy bundle',
    )
    ax.tick_params(axis='x', labelrotation=70, labelsize=8)
    ax.tick_params(axis='y', labelsize=6)
    fig.tight_layout()
    fig.savefig(OUTPUT_ROOT / f'raw_{depth}_feature_heatmap.png', dpi=180, bbox_inches='tight')
    plt.show()

## PCA of each unstandardized space

PCA centers each feature but does not rescale it. Consequently, the loadings show which raw FORUM axes drive the largest differences among policy bundles. Every bundle is plotted; only cluster medoids are labelled to avoid an unreadable field of 90 labels.

In [ ]:
coordinate_parts = []
loading_parts = []
fig, axes = plt.subplots(1, 2, figsize=(19, 8))
for axis, depth in zip(axes, DEPTHS):
    matrix = matrices[depth]
    pca = PCA(n_components=min(10, matrix.shape[1]), random_state=20260902)
    values = pca.fit_transform(matrix)
    coordinates = policy_metadata.copy()
    coordinates['depth'] = depth
    coordinates['cluster'] = spaces[depth]['labels'].astype(str)
    coordinates['pc1'] = values[:, 0]
    coordinates['pc2'] = values[:, 1]
    coordinate_parts.append(coordinates)
    loadings = pd.DataFrame(
        pca.components_.T, index=matrix.columns,
        columns=[f'PC{i + 1}' for i in range(pca.n_components_)],
    )
    loadings['depth'] = depth
    loadings['feature'] = loadings.index
    loading_parts.append(loadings.reset_index(drop=True))

    sns.scatterplot(
        data=coordinates, x='pc1', y='pc2', hue='cluster', style='interface',
        palette='tab10', s=80, alpha=.85, ax=axis,
    )
    labels = spaces[depth]['labels']
    medoids = []
    for cluster in np.unique(labels):
        members = np.flatnonzero(labels == cluster)
        local = spaces[depth]['distance'][np.ix_(members, members)]
        medoids.append(members[np.argmin(local.sum(axis=1))])
    for index in medoids:
        axis.annotate(
            coordinates.policy_id.iloc[index],
            (coordinates.pc1.iloc[index], coordinates.pc2.iloc[index]), fontsize=7,
        )
    axis.set(
        title=f'{depth}: PCA of raw FORUM feature profiles',
        xlabel=f"PC1 ({pca.explained_variance_ratio_[0]:.1%})",
        ylabel=f"PC2 ({pca.explained_variance_ratio_[1]:.1%})",
    )
    axis.legend(frameon=False, fontsize=7, loc='best')
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / 'raw_pca.png', dpi=180, bbox_inches='tight')
plt.show()

pca_coordinates = pd.concat(coordinate_parts, ignore_index=True)
pca_loadings = pd.concat(loading_parts, ignore_index=True)
for depth in DEPTHS:
    panel = pca_loadings[pca_loadings.depth.eq(depth)].copy()
    print(f'Largest raw loadings: {depth}')
    for component in ('PC1', 'PC2'):
        display(panel.assign(abs_loading=panel[component].abs()).nlargest(12, 'abs_loading')[
            ['feature', component, 'abs_loading']
        ])

## What is in each cluster?

Cluster composition tables separate the ordering rule from the interface transformations. This helps distinguish clusters driven by chronological/upvote/model orderings from clusters driven mainly by reply handling or pinning.

In [ ]:
clustered_parts = []
for depth in DEPTHS:
    clustered = policy_metadata.copy()
    clustered['depth'] = depth
    clustered['cluster'] = spaces[depth]['labels']
    clustered_parts.append(clustered)
    print(f'Cluster sizes and interface composition: {depth}')
    display(clustered.groupby('cluster').agg(
        bundles=('policy_id', 'size'), orderings=('ordering', 'nunique'),
        reply_modes=('reply_mode', lambda values: ', '.join(sorted(set(values)))),
        pinned_bundles=('pinned', 'sum'),
    ))
    display(pd.crosstab(clustered.ordering, clustered.cluster))
clustered_policies = pd.concat(clustered_parts, ignore_index=True)

## How similar are the top-10 and full spaces?

Distance correlation asks whether the same pairs of bundles are near one another in both spaces. Adjusted Rand agreement compares the selected cluster partitions even when their selected `k` differs. The final table identifies bundles whose neighbourhood relationships change most between depths.

In [ ]:
top_distance = spaces['top10']['distance']
full_distance = spaces['full']['distance']
upper = np.triu_indices(len(policy_metadata), k=1)
distance_rho = spearmanr(top_distance[upper], full_distance[upper]).statistic
partition_ari = adjusted_rand_score(spaces['top10']['labels'], spaces['full']['labels'])
print(f'Spearman correlation between pairwise distances: {distance_rho:.3f}')
print(f'Adjusted Rand agreement between selected partitions: {partition_ari:.3f}')
display(pd.crosstab(
    pd.Series(spaces['top10']['labels'], name='top10_cluster'),
    pd.Series(spaces['full']['labels'], name='full_cluster'),
))

neighbourhood_change = np.mean(np.abs(top_distance - full_distance), axis=1)
changed = policy_metadata.copy()
changed['mean_absolute_distance_change'] = neighbourhood_change
changed['top10_cluster'] = spaces['top10']['labels']
changed['full_cluster'] = spaces['full']['labels']
display(changed.nlargest(20, 'mean_absolute_distance_change')[
    ['policy_id', 'ordering', 'reply_mode', 'pinned', 'top10_cluster',
     'full_cluster', 'mean_absolute_distance_change']
])

In [ ]:
nearest_rows = []
for depth in DEPTHS:
    distance = spaces[depth]['distance']
    for index, policy_id in enumerate(policy_metadata.policy_id):
        neighbours = np.argsort(distance[index])
        neighbours = neighbours[neighbours != index][:3]
        for neighbour_rank, neighbour in enumerate(neighbours, start=1):
            nearest_rows.append({
                'depth': depth, 'policy_id': policy_id,
                'neighbour_rank': neighbour_rank,
                'neighbour_policy_id': policy_metadata.policy_id.iloc[neighbour],
                'euclidean_distance': distance[index, neighbour],
            })
nearest_neighbours = pd.DataFrame(nearest_rows)
display(nearest_neighbours[nearest_neighbours.neighbour_rank.eq(1)].head(30))

## Optional UMAP

If `umap-learn` is available, this adds a local-neighbourhood view for both raw spaces. UMAP is not used to choose clusters: distances and apparent gaps in a UMAP plot are sensitive to its neighbourhood and minimum-distance settings.

In [ ]:
try:
    import umap
except ImportError:
    print('Optional UMAP skipped: umap-learn is not installed. PCA remains the reproducible main visualization.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    for axis, depth in zip(axes, DEPTHS):
        embedding = umap.UMAP(
            n_components=2, n_neighbors=15, min_dist=.15,
            metric='euclidean', random_state=20260902,
        ).fit_transform(matrices[depth])
        frame = policy_metadata.copy()
        frame['umap1'], frame['umap2'] = embedding[:, 0], embedding[:, 1]
        frame['cluster'] = spaces[depth]['labels'].astype(str)
        sns.scatterplot(
            data=frame, x='umap1', y='umap2', hue='cluster', style='interface',
            palette='tab10', s=80, alpha=.85, ax=axis,
        )
        axis.set_title(f'{depth}: optional UMAP of raw FORUM profiles')
        axis.legend(frameon=False, fontsize=7)
    fig.tight_layout()
    fig.savefig(OUTPUT_ROOT / 'raw_umap.png', dpi=180, bbox_inches='tight')
    plt.show()

In [ ]:
feature_means.to_csv(OUTPUT_ROOT / 'policy_feature_mean_forum_outcomes.csv', index=False)
cluster_selection.to_csv(OUTPUT_ROOT / 'cluster_selection.csv', index=False)
clustered_policies.to_csv(OUTPUT_ROOT / 'cluster_membership.csv', index=False)
pca_coordinates.to_csv(OUTPUT_ROOT / 'pca_coordinates.csv', index=False)
pca_loadings.to_csv(OUTPUT_ROOT / 'pca_loadings.csv', index=False)
nearest_neighbours.to_csv(OUTPUT_ROOT / 'nearest_neighbours.csv', index=False)
feature_dispersion.to_csv(OUTPUT_ROOT / 'raw_feature_dispersion.csv')

for depth in DEPTHS:
    matrices[depth].to_csv(OUTPUT_ROOT / f'{depth}_raw_forum_matrix.csv')
    pd.DataFrame(
        spaces[depth]['distance'], index=policy_metadata.policy_id,
        columns=policy_metadata.policy_id,
    ).to_csv(OUTPUT_ROOT / f'{depth}_euclidean_distance.csv')
print(f'Saved reusable tables to {OUTPUT_ROOT.relative_to(REPO_ROOT)}')

figure_inventory = pd.DataFrame({
    'figure': sorted(path.name for path in OUTPUT_ROOT.glob('*.png'))
})
display(figure_inventory)
print(f'{len(figure_inventory)} saved figures')